# Exploratory Visualization: Olympic Games and Displacement

Interactive Altair visualizations exploring reported population displacement linked to Olympic host cities.

## Data Sources

- **COHRE 2007** — *Fair Play for Housing Rights: Mega-Events Olympic Games and Housing Rights*
- **Comitê Popular Rio (2014)** — *Dossiê Megaeventos e Violações dos Direitos Humanos no Rio de Janeiro*
- **IOC records** — host city / Games year reference

## How the dataset was built

No public dataset directly measures Olympic-related displacement. We assembled `data/cohre_displacement_estimates.csv` by:
1. Extracting **point estimates** from the COHRE 2007 report (Seoul, Atlanta, Beijing; placeholders for Barcelona/Athens/London).
2. Adding **Rio 2016** from the Comitê Popular Rio dossier (reported in *families*, not people).
3. Recording **provenance** (`source_short`, `source_title`, `notes`) for each row so figures stay traceable.
4. Flagging **`unknown`** values where no canonical figure could be extracted.
5. Joining with `olympic_housing_event_spine.csv` for timeline alignment.

**Important:** these are NGO archival estimates with inconsistent units and methodology across cities. They are **context evidence**, not official statistics.

In [8]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import altair as alt

alt.data_transformers.disable_max_rows()


def load_displacement() -> pd.DataFrame:
    df = pd.read_csv('data/cohre_displacement_estimates.csv')
    # 'unknown' -> NaN so we can plot only documented estimates.
    df['estimated_displaced_people'] = pd.to_numeric(
        df['estimated_displaced_people'], errors='coerce'
    )
    # Normalise Rio (reported in families) to approximate people, ~3.5 per family,
    # purely for cross-city scale comparability. Original unit preserved separately.
    df['estimated_displaced_people_norm'] = np.where(
        df['estimate_unit'] == 'families',
        df['estimated_displaced_people'] * 3.5,
        df['estimated_displaced_people'],
    )
    df['has_estimate'] = df['estimated_displaced_people'].notna()
    return df


def load_event_spine() -> pd.DataFrame:
    return pd.read_csv('data/olympic_housing_event_spine.csv')

In [7]:
# Load and prep data
displacement = load_displacement()
spine = load_event_spine()

# Join for context (games_year exists in both)
displacement = displacement.merge(
    spine[['games_year', 'event_label']], on='games_year', how='left'
)

known = displacement[displacement['has_estimate']].copy()
unknown = displacement[~displacement['has_estimate']].copy()

print('✓ Data loaded')
print('Rows total:', len(displacement))
print('Rows with numeric estimate:', len(known))
print('Rows flagged unknown:', len(unknown))
print('Sources:', sorted(displacement['source_short'].unique()))

✓ Data loaded
Rows total: 7
Rows with numeric estimate: 4
Rows flagged unknown: 3
Sources: ['COHRE 2007', 'COHRE+case studies', 'Comite Popular Rio 2014']


## Hypothesis: Olympic Hosting and Reported Displacement

**Hypothesis:** Hosting the Summer Olympic Games is associated with substantial reported population displacement.

In [11]:
# Visual 1: Reported displacement by host city (documented estimates only)
d1 = known.copy()
d1['city_label'] = d1['host_city'] + ' ' + d1['games_year'].astype(str)

v1 = alt.Chart(d1).mark_bar(color='tomato').encode(
    x=alt.X('estimated_displaced_people_norm:Q',
            title='Estimated Displaced'),
    y=alt.Y('city_label:N', title='', sort='-x'),
    tooltip=[
        'host_city:N', 'games_year:Q',
        alt.Tooltip('estimated_displaced_people:Q', format=',', title='Reported value'),
        'estimate_unit:N', 'source_short:N'
    ]
).properties(
    width=700, height=300,
    title='Visual 1: Reported Displacement by Host City (documented estimates)'
)
v1.display()

alt.Chart(...)

### Visual 1: Reported Displacement by Host City

**What's informative about this view:**
- Communicates the scale gap between hosts (Beijing and Seoul dominate; Atlanta and Rio look small in comparison).
- Anchors the hypothesis in concrete archival numbers, not abstractions.
- Tooltips expose the *original unit and source* so the audience sees provenance.

**What could be improved about this view:**
- Use a stackbar to really show the total amount of displacement caused by the Olympics (cf Visual 2)
- Going more granular to understand the circumstances of the displacements
- Getting more data of more host cities (Athens, Barcelona...)



In [ ]:
# Visual 2: Cumulative documented displacement, stacked by host city
d2 = known.copy()
d2['city_label'] = d2['host_city'] + ' ' + d2['games_year'].astype(str)
d2['stack_group'] = 'Total documented displacement'

missing_cities = unknown['host_city'].tolist()
missing_note = (
    f"Missing / undocumented in this dataset: {', '.join(missing_cities)}"
    if missing_cities else "All host cities have documented estimates."
)

total_documented = int(d2['estimated_displaced_people_norm'].sum())

stacked = alt.Chart(d2).mark_bar().encode(
    x=alt.X('stack_group:N', title='', axis=alt.Axis(labelAngle=0)),
    y=alt.Y('estimated_displaced_people_norm:Q',
            title='People displaced (Rio normalised from families)',
            stack='zero'),
    color=alt.Color('city_label:N',
                    sort=alt.SortField('estimated_displaced_people_norm', order='descending'),
                    legend=alt.Legend(title='Host city (Games year)')),
    order=alt.Order('estimated_displaced_people_norm:Q', sort='descending'),
    tooltip=[
        'host_city:N', 'games_year:Q',
        alt.Tooltip('estimated_displaced_people:Q', format=',', title='Reported value'),
        'estimate_unit:N', 'source_short:N'
    ]
).properties(
    width=350, height=500,
    title=alt.TitleParams(
        text='Over 2 million displacements since 1988',
        subtitle=[
            f'Total across documented hosts: ~{total_documented:,} people',
            missing_note,
        ],
    )
)

stacked.display()


alt.Chart(...)

### Visual 2: Cumulative Documented Olympic Displacement (stacked bar)

**What's informative about this view:**
- Communicates a single headline number: the **total documented** displacement across all Olympic Games we have data for.
- Each color segment shows how much each host city contributes to that total.
- The subtitle explicitly names the hosts where we are **still missing data** (Barcelona, Athens, London), so the audience never reads the total as exhaustive.

**What could be improved about this view:**
- A single stacked column flattens chronology — Visual 1 should be read alongside for the per-host view.
- Rio is normalised from *families* to *people* (×3.5), which is a rough conversion.
- Filling in the missing hosts could meaningfully shift the total upward.

## Conclusion

**Hypothesis:** Hosting the Summer Olympic Games is associated with substantial reported displacement, but the magnitude varies dramatically across hosts depending on governance context, urban-renewal scope, and documentation quality.

The data supports the hypothesis qualitatively: every documented host shows non-trivial displacement, and the magnitude varies by orders of magnitude (15k in Atlanta vs ~1.5M in Beijing). However, the dataset is **archival, sparse, and methodologically heterogeneous**, and several hosts remain undocumented.
